# V62 - Per-module LightGBM

Pre-registered plan: gradient boosting is the one model family with lineage precedent (v3.1 calibrated blend reached public .65639 from these same data) that my loop has not tested grouped. Per-module LightGBM, 500 trees, regularized leaves for thin signal, on word TF-IDF plus assessment. Fresh seed, grouped folds, floor 0.003 for a model-class change with no negative fold. If it clears, it becomes the new base for composition. If not, tree boosting joins the closed list on raw features.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 404
N_SPLITS = 5
GATE_FLOOR = 0.003
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}
PARAMS = dict(objective="regression", n_estimators=500, learning_rate=0.05, num_leaves=31,
                min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                verbose=-1, deterministic=True, seed=SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v62")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| lgb:", lgb.__version__)

## Grouped OOF: LightGBM vs Ridge on identical features

Same sparse word matrix plus dense assessment for both, so only the learner differs.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
X = hstack([W, csr_matrix(A)]).tocsr()
X_tr = X[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_r = np.zeros_like(Y)
oof_g = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(X_tr[tr].toarray(), Y[tr, j])
        oof_r[va, j] = m.predict(X_tr[va].toarray())
        g = lgb.LGBMRegressor(**PARAMS)
        g.fit(X_tr[tr], Y[tr, j])
        oof_g[va, j] = g.predict(X_tr[va])
    fids[va] = fi + 1
    print(f"fold {fi + 1}: ridge={ndcg5(Y[va], oof_r[va]).mean():.5f} lgbm={ndcg5(Y[va], oof_g[va]).mean():.5f}", flush=True)

base = ndcg5(Y, oof_r)
d = ndcg5(Y, oof_g) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_g[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"ridge={base.mean():.5f} lgbm={(base + d).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT LightGBM as base" if (lo > GATE_FLOOR and wf >= 0) else "KEEP Ridge; boosting adds nothing on raw features")